# Automatidata: estimar la tarifa de un taxi antes del viaje

**Curso 4 del certificado, proyecto de regresión lineal múltiple.**

La Comisión de Taxis y Limusinas de Nueva York quiere enseñar al pasajero una tarifa
estimada antes de que se suba. Este cuaderno construye ese estimador y, sobre todo, dice
cuánto se le puede creer.

La pregunta lleva una condición que decide el proyecto entero: **antes del viaje**. La
distancia recorrida y la duración son los dos mejores predictores de la tarifa y ninguno
de los dos existe todavía en el momento de dar el precio.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "projects" / "curso4").is_dir():      # run from anywhere in the project
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "projects"))
sys.path.insert(0, str(ROOT / "projects" / "curso4" / "automatidata" / "02_scripts"))

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

from automatidata_regression import (
    add_features, add_leaky_pair_means, add_pair_means, fit, load_and_clean, score,
    vif_table,
)
from common import Results

pd.set_option("display.width", 120)
print("listo")

listo


## 1. Los datos, y qué hubo que arreglar

Se descartan los viajes físicamente imposibles y se recortan los extremos al percentil
99,5 en vez de borrarlos, para no cambiar en silencio la población que el modelo describe.

In [2]:
results = Results("automatidata", "Automatidata, cuaderno")
df = add_features(load_and_clean(results))
df[["fare_amount", "trip_distance", "duration", "airport_flat", "rush_hour"]].head()


1. Los datos, y qué hubo que arreglar
  filas en el CSV: 22699
  columnas: 17
  filas duplicadas: 0
  celdas vacías: 0
  viajes a 52,00 dólares exactos: 514
  de esos, con tarifa plana de aeropuerto: 513
  viajes con tarifa plana en total: 513
  descartados por fare_amount <= 0: 20
  descartados por duration <= 0: 27
  descartados por trip_distance <= 0: 148
  filas descartadas en total: 165
  filas que quedan: 22534
  tope del 99.5% para fare_amount: 55.67
    filas recortadas en fare_amount: 113
  tope del 99.5% para duration: 73.18
    filas recortadas en duration: 113
  tope del 99.5% para trip_distance: 20.55
    filas recortadas en trip_distance: 113
  tarifa media: 12.85
  tarifa mediana: 9.5
  distancia mediana (millas): 1.63
  duración mediana (minutos): 11.23


           fare_amount  trip_distance   duration  airport_flat  rush_hour
24870114          13.0           3.34  14.066667             0          0
35634249          16.0           1.80  26.500000             0          0
106203690          6.5           1.00   7.200000             0          1
38942136          20.5           3.70  30.250000             0          0
30841670          16.5           4.37  16.716667             0          0


## 2. El hallazgo que condiciona el modelo: la tarifa plana del aeropuerto

514 viajes cuestan exactamente 52,00 dólares, y 513 de ellos llevan el código de tarifa 2.
No es una coincidencia de precios: es un precio fijo.

In [3]:
flat = df[df.airport_flat.eq(1)]
metered = df[df.airport_flat.eq(0)]

for name, part in [("taximetro", metered), ("tarifa plana", flat)]:
    slope = np.polyfit(part.trip_distance, part.fare_amount, 1)[0]
    print(f"{name:<14} {len(part):>6} viajes   {abs(slope):.2f} $ por milla")

taximetro       22037 viajes   2.86 $ por milla
tarifa plana      497 viajes   0.00 $ por milla


![Tarifa contra distancia](03_figures/02_tarifa_vs_distancia.png)

## 3. Separar entrenamiento y prueba, y solo entonces construir las medias por ruta

La distancia y la duración medias de cada pareja origen-destino sí se conocen al reservar,
y son el sustituto legítimo de la distancia real.

**Se calculan solo con los datos de entrenamiento.** Calcularlas sobre el dataset completo
mete información de los viajes de prueba dentro del entrenamiento, y más abajo se mide
exactamente cuánto engaña eso.

In [4]:
train, test = train_test_split(df, test_size=0.25, random_state=42)
train, test = add_pair_means(train.copy(), test.copy(), results)

print(f"entrenamiento {len(train)}   prueba {len(test)}")
print(f"rutas nuevas en prueba: {(~test.route_seen).sum()} ({(~test.route_seen).mean():.1%})")

  parejas origen-destino aprendidas: 3679
  viajes de prueba por una ruta nunca vista: 512
    en porcentaje: 9.1
entrenamiento 16900   prueba 5634
rutas nuevas en prueba: 512 (9.1%)


## 4. Multicolinealidad: la distancia real y la media de la ruta miden lo mismo

Un factor de inflación de la varianza por encima de 10 significa que la variable se explica
casi entera con las demás, y que su coeficiente deja de poder interpretarse por separado.

In [5]:
everything = ["trip_distance", "duration", "mean_distance", "mean_duration",
              "passenger_count", "rush_hour", "airport_flat"]
print(vif_table(train[everything]).to_string(index=False, float_format=lambda v: f"{v:.2f}"))

       variable   vif
  mean_distance 32.17
  trip_distance 28.68
  mean_duration  7.73
       duration  4.50
   airport_flat  1.64
      rush_hour  1.01
passenger_count  1.00


## 5. Tres modelos, y solo uno se puede entregar

- **A**: con la distancia y la duración reales del viaje.
- **B**: medias de ruta calculadas sobre todo el dataset, que es el atajo habitual.
- **C**: las mismas medias, calculadas solo con el entrenamiento.

In [6]:
after = ["trip_distance", "duration", "rush_hour", "airport_flat"]
before = ["mean_distance", "mean_duration", "rush_hour", "airport_flat"]

leaky = add_leaky_pair_means(df)
train_b, test_b = train_test_split(leaky, test_size=0.25, random_state=42)

runs = [
    ("A  distancia real del viaje", fit(train, after), test, after),
    ("B  medias de todo el dataset", fit(train_b, before), test_b, before),
    ("C  medias solo del entrenamiento", fit(train, before), test, before),
]
for name, model, frame, features in runs:
    s = score(model, frame, features)
    print(f"{name:<34} R2 {s['r2']:.4f}   RMSE {s['rmse']:5.2f} $   MAE {s['mae']:.2f} $")

A  distancia real del viaje        R2 0.9461   RMSE  2.41 $   MAE 0.82 $
B  medias de todo el dataset       R2 0.8847   RMSE  3.53 $   MAE 2.06 $
C  medias solo del entrenamiento   R2 0.6746   RMSE  5.93 $   MAE 3.21 $


El modelo A predice casi perfecto y **no se puede desplegar**, porque usa datos que no
existen al reservar. Entre B y C hay 0,21 de R² y 2,40 dólares de error: esa es la
diferencia entre el número que sale en un cuaderno y el que se ve en producción.

## 6. El modelo que se entrega

In [7]:
model = fit(train, before)
print(model.summary().tables[1])
print(f"\nR2 entrenamiento {model.rsquared:.4f}   R2 ajustado {model.rsquared_adj:.4f}")

                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             2.6516      0.054     48.910      0.000       2.545       2.758
mean_distance     1.8704      0.017    107.435      0.000       1.836       1.905
mean_duration     0.3162      0.005     57.859      0.000       0.306       0.327
rush_hour         0.2660      0.057      4.677      0.000       0.155       0.377
airport_flat      3.2971      0.225     14.684      0.000       2.857       3.737

R2 entrenamiento 0.8923   R2 ajustado 0.8923


![Coeficientes](03_figures/04_coeficientes.png)

## 7. Dónde falla: la ruta que no ha visto nunca

El error medio del conjunto esconde dos poblaciones muy distintas.

In [8]:
scored = score(model, test, before)
error = scored["residuals"].abs()
seen = test.route_seen

for name, mask in [("ruta conocida", seen), ("ruta nueva", ~seen)]:
    print(f"{name:<16} {mask.sum():>5} viajes   error medio {error[mask].mean():5.2f} $")
print(f"\nla ruta nueva es {error[~seen].mean() / error[seen].mean():.1f} veces peor")

ruta conocida     5122 viajes   error medio  2.42 $
ruta nueva         512 viajes   error medio 11.11 $

la ruta nueva es 4.6 veces peor


![Error por familiaridad de la ruta](03_figures/05_rutas_nuevas.png)

## 8. Los supuestos, sin maquillar

Dos de los cuatro no se cumplen, y la figura de residuos enseña por qué: las rutas nuevas
reciben todas la misma predicción y forman una franja vertical, y los viajes de tarifa
plana forman una diagonal perfecta porque su tarifa real es siempre 52.

In [9]:
from scipy import stats

residuals = scored["residuals"]
print(f"media de los residuos     {residuals.mean():.4f}")
print(f"asimetria                 {stats.skew(residuals):.3f}   (0 seria simetrico)")
print(f"curtosis                  {stats.kurtosis(residuals):.3f}   (0 seria normal)")

helper = sm.OLS(residuals ** 2, sm.add_constant(scored["predicted"])).fit()
print(f"varianza constante: p = {helper.f_pvalue:.3g}   (bajo = no se cumple)")

media de los residuos     0.8692
asimetria                 3.494   (0 seria simetrico)
curtosis                  18.073   (0 seria normal)
varianza constante: p = 1.74e-06   (bajo = no se cumple)


![Residuos](03_figures/03_residuos.png)

## 9. Conclusión

El modelo estima la tarifa antes del viaje con un error medio de **2,42 dólares en las
rutas conocidas**, que son el 90,9 % de los casos, y de **11,11 dólares en las rutas
nuevas**, que son el 9,1 % restante.

Se recomienda desplegarlo para rutas conocidas anunciando ese margen, y arreglar el
respaldo de ruta nueva antes de ampliar la cobertura: hoy responde 12,68 dólares a
cualquier ruta desconocida, que es la media de la ciudad, y estimar la distancia entre las
dos zonas sería mejor que eso sin necesidad de otro modelo.

El informe completo está en `04_reports/executive_summary.md` y todos los números
publicados en `04_reports/model_results.json`.